In [9]:
from collections import Counter, defaultdict
import random
class Classifier:

    def fit(self, X, y):
        raise NotImplementedError

    def predict(self, X):
        raise NotImplementedError

    def score(self, X, y):
        predictions = self.predict(X)
        correct = sum(
            actual == predicted
            for actual, predicted in zip(y, predictions)
        )
        return correct / len(y)

    def error(self, X, y):
        return 1 - self.score(X, y)

In [10]:
class MajorityVoteClassifier(Classifier):

    def fit(self, X, y):
        self.label = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [self.label] * len(X)

In [11]:
class MemorizerClassifier(Classifier):

    def fit(self, X, y):
        self.memory = dict(zip(map(tuple, X), y))
        self.labels = list(set(y))
        return self

    def predict(self, X):
        return [
            self.memory.get(tuple(x), random.choice(self.labels))
            for x in X
        ]


In [12]:
class DecisionStumpClassifier(Classifier):

    def __init__(self, feature_index=0):
        self.feature_index = feature_index

    def fit(self, X, y):
        groups = defaultdict(list)

        for x, label in zip(X, y):
            groups[x[self.feature_index]].append(label)

        self.rules = {
            value: Counter(labels).most_common(1)[0][0]
            for value, labels in groups.items()
        }

        self.default = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [
            self.rules.get(x[self.feature_index], self.default)
            for x in X
        ]

In [13]:
X_train = [
    ["Y", "N", "N"],
    ["N", "Y", "N"],
    ["Y", "Y", "N"],
    ["Y", "N", "Y"],
    ["N", "Y", "Y"],
    ["Y", "Y", "Y"],
    ["N", "N", "N"],
    ["N", "N", "Y"]
]

y_train = ["-", "-", "+", "+", "+", "+", "-", "-"]


X_test = [
    ["Y", "Y", "N"],
    ["N", "N", "Y"],
    ["Y", "N", "N"],
    ["N", "Y", "Y"]
]

y_test = ["+", "-", "-", "+"]

In [15]:
models = {
    "majority vote": MajorityVoteClassifier(),
    "memorizer": MemorizerClassifier(),
    "decision stump": DecisionStumpClassifier(feature_index=0)
}

for name, model in models.items():

    model.fit(X_train, y_train)

    print(name)
    print("training ccore:", model.score(X_train, y_train))
    print("test score:", model.score(X_test, y_test))
    print()

majority vote
training ccore: 0.5
test score: 0.5

memorizer
training ccore: 1.0
test score: 1.0

decision stump
training ccore: 0.75
test score: 0.5

